# Notebook 4 - Segmentation and Insights
Alok Chauhan - 251810700318
Aman Kumar - 251810700231

segmentation means dividing messages into groups
and checking what percentage of each group is spam
this helps us find patterns and make rules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv("outputs/spam_cleaned.csv")
spam = data[data["label"] == "spam"]
ham  = data[data["label"] == "ham"]

# overall spam rate - this is our baseline
overall_spam = data["label_num"].mean() * 100
print("overall spam rate:", round(overall_spam, 1), "%")
print("total messages:", len(data))

group 1 - dividing messages by how long they are
short messages vs long messages
which length has more spam?

In [ ]:
# dividing messages into length groups
data["length_group"] = pd.cut(
    data["char_count"],
    bins=[0, 50, 100, 160, 300, 9999],
    labels=["0-50 chars", "51-100", "101-160", "161-300", "300+"]
)

# checking spam rate in each group
group1 = data.groupby("length_group", observed=True).agg(
    total      = ("label", "count"),
    spam_count = ("label_num", "sum")
)
group1["spam_rate"] = (group1["spam_count"] /
                       group1["total"] * 100).round(1)

print("spam rate by message length:")
print(group1)

wow! messages between 101-160 chars have 40% spam rate
that is 3 times higher than the overall average of 12.6%
this makes sense because spammers try to fit in one SMS

In [ ]:
# group 2 - how many spam signals does the message have?
group2 = data.groupby("spam_signals").agg(
    total      = ("label", "count"),
    spam_count = ("label_num", "sum")
)
group2["spam_rate"] = (group2["spam_count"] /
                       group2["total"] * 100).round(1)

print("spam rate by number of signals:")
print(group2)
print("\nbig finding! 3 or more signals = 100% spam!")

group 3 - does the message have a phone number?
phone numbers should be a very strong spam signal

In [ ]:
# group 3 - phone number
group3 = data.groupby("has_phone").agg(
    total      = ("label", "count"),
    spam_count = ("label_num", "sum")
)
group3["spam_rate"] = (group3["spam_count"] /
                       group3["total"] * 100).round(1)
group3.index = ["no phone number", "has phone number"]

print("spam rate by phone number:")
print(group3)

group 4 - exclamation marks
do more exclamation marks mean more spam?

In [ ]:
# group 4 - exclamation marks
data["excl_group"] = pd.cut(
    data["exclamation"],
    bins=[-1, 0, 1, 2, 9999],
    labels=["0 marks", "1 mark", "2 marks", "3 or more"]
)
group4 = data.groupby("excl_group", observed=True).agg(
    total      = ("label", "count"),
    spam_count = ("label_num", "sum")
)
group4["spam_rate"] = (group4["spam_count"] /
                       group4["total"] * 100).round(1)

print("spam rate by exclamation marks:")
print(group4)

making charts for all 4 groups

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Spam Rate by Different Groups", fontsize=14)

# helper function to make each chart
def make_chart(ax, group, title):
    labels = [str(i) for i in group.index]
    rates  = group["spam_rate"].values
    totals = group["total"].values

    # red for high spam, orange for medium, green for low
    colors = []
    for r in rates:
        if r > 50:
            colors.append("#E74C3C")
        elif r > 20:
            colors.append("#e67e22")
        else:
            colors.append("#2ECC71")

    bars = ax.bar(labels, rates, color=colors)

    # dotted line showing overall average
    ax.axhline(overall_spam, color="blue",
               linestyle="--", lw=2,
               label=f"overall avg: {overall_spam:.1f}%")

    # adding numbers on top of bars
    for bar, rate, total in zip(bars, rates, totals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 1,
                f"{rate:.0f}%\n({total})",
                ha="center", fontsize=9)

    ax.set_title(title)
    ax.set_ylabel("spam rate %")
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=10)

make_chart(axes[0,0], group1, "1. By Message Length")
make_chart(axes[0,1], group2, "2. By Spam Signal Score")
make_chart(axes[1,0], group3, "3. Phone Number Present?")
make_chart(axes[1,1], group4, "4. Exclamation Marks")

plt.tight_layout()
plt.savefig("outputs/04_segmentation.png", dpi=120)
plt.show()
print("chart saved!")

## top 5 findings from all notebooks

1. phone numbers - 99.7% of messages with a phone number are spam
   rule: flag any message with a 10+ digit number

2. URLs - messages with URLs are 57 times more likely to be spam
   rule: flag messages with web links for review

3. message length - spam messages are almost 2 times longer than ham
   spam median is 149 chars, ham median is only 52 chars

4. spam signals score - 3 or more signals means 100% spam
   rule: block any message with score 3 or above

5. prize and win words - 35 times more common in spam than ham
   rule: flag messages with prize, win, cash, free together

## our rule pipeline
if message has phone number → block (99.7% accurate)
if message has URL → send for review
if spam_signals score is 3 or more → block
if spam_signals score is 2 → send for review
otherwise → pass through